# Module 1 · Lab 1.1 — Prompting Techniques & Structured Outputs

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---

**Scenario.** Retail + corporate banking (customer support, transaction categorization, structured intake, loan assessment, debt management). The lab demonstrates the difference between sloppy prompts and precisely-crafted ones using `ChatPromptTemplate`.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Build prompts using LangChain's `ChatPromptTemplate` from cell 1 — no raw f-string prompts.
2. Build a chat prompt from system, human and AI messages, with a `MessagesPlaceholder` for conversation history — and say when a raw provider SDK is enough.
3. Understand **tokens** and **context windows** — the units an LLM actually sees
4. Provide clear context, role, and constraints to control model output.
5. Apply **zero-shot** and **few-shot** prompting to common banking tasks and choose between them.
6. Add **directional stimulus** to focus the model on specific aspects of input.
7. Use **Chain-of-Thought (CoT)** to make multi-step reasoning auditable.
8. Apply a **Tree-of-Thought (ToT)-style** prompt to explore multiple hypotheses before deciding.
9. Decide which technique to use given a task's complexity, stakes, and latency budget.
10. Force LLM output into a **Pydantic schema** and validate it
11. Read a poorly-crafted prompt's output, diagnose the issue, and rewrite the prompt.
12. Set up a reusable `prompt | llm` chain that can be composed with parsers (Lab 2.1) and memory (Lab 4.2).

*After the session:*
1. Apply a **persona** to set tone, expertise level, and audience.
2. Enforce a **structured output template** (JSON, table, fixed-section) that downstream code can parse.
3. Scale few-shot examples for production using `FewShotChatMessagePromptTemplate` instead of inlining examples in a string.

**Tools.** LangChain v1 · `langchain-openai` · `gpt-4.1-mini` (non-reasoning, supports temperature) · `gpt-5-mini` (reasoning model, for structured output).

> 🗺️ **Workshop run-list.**
> **Live in session:** §1–4 — set-up, messages and templates, tokens, crafting precise prompts; §6.1–6.2 zero- and few-shot; §7 directional stimulus; §9.1 Chain-of-Thought; §10.1 Tree-of-Thought (the designated cut if time is short); §11 choosing a pattern; §12 structured output ⭐; §14
> **After the session:** §5 persona, §6.3–6.7 few-shot at scale and four more zero/few-shot use cases, §8 template, §9.2 and §10.2 second use cases, §13 challenge and stretch exercises, interview preparation — kept in full so you can finish at your own pace (marked ⏭️).

## 1. Set-up

In [1]:
# 📦 Packages this notebook needs. On the workshop VM they are pre-installed from
# the workshop's requirements.txt, so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU langchain langchain-core langchain-openai openai pydantic python-dotenv tiktoken pandas

from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_PACKAGES = [
    "langchain",
    "langchain-core",
    "langchain-openai",
    "openai",
    "pydantic",
    "python-dotenv",
    "tiktoken",
    "pandas",
]
for package_name in NOTEBOOK_PACKAGES:
    try:
        print(f"{package_name:<26} {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name:<26} NOT INSTALLED — uncomment the !pip line above")

langchain                  1.4.3
langchain-core             1.6.6
langchain-openai           1.6.6
openai                     2.54.0
pydantic                   2.12.5
python-dotenv              1.2.3
tiktoken                   0.14.0
pandas                     2.3.3


In [2]:
import os

# Where to create each key. The trainer supplies OPENAI_API_KEY on the workshop VM;
# you create any other key yourself and add it to the workshop's .env file as KEY_NAME=value.
KEY_SIGNUP_PAGES = {
    "OPENAI_API_KEY": "supplied on the workshop VM (elsewhere: https://platform.openai.com/api-keys)",
    "TAVILY_API_KEY": "https://app.tavily.com (free tier)",
    "GROQ_API_KEY": "https://console.groq.com/keys (free tier)",
    "WEATHER_API_KEY": "https://www.weatherapi.com/signup.aspx (free tier)",
    "LANGSMITH_API_KEY": "https://smith.langchain.com -> Settings -> API Keys (free tier)",
    "ANTHROPIC_API_KEY": "https://console.anthropic.com/settings/keys",
    "GEMINI_API_KEY": "https://aistudio.google.com/app/apikey",
}


def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere).

    Prints one line per key, then stops with instructions if a required key is missing.

    Args:
        required: the environment-variable names this lab needs.
    """
    try:
        from google.colab import userdata          # running on Google Colab
        for key_name in required:
            try:
                os.environ[key_name] = userdata.get(key_name)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # the workshop VM, or any local machine
        from dotenv import find_dotenv, load_dotenv
        load_dotenv(find_dotenv(usecwd=True))        # searches upward for the workshop's .env file
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for key_name in required:
        print(f"   {key_name:<22} {'✅' if os.environ.get(key_name) else '❌ MISSING'}")
    missing_keys = [key_name for key_name in required if not os.environ.get(key_name)]
    if missing_keys:
        instructions = "\n".join(
            f"  {key_name}: {KEY_SIGNUP_PAGES.get(key_name, 'see README.md in the workshop folder')}"
            for key_name in missing_keys)
        raise RuntimeError(
            "This lab needs API keys that are not set yet. Create each one, add it to\n"
            "the workshop's .env file as KEY_NAME=value, then re-run this cell:\n" + instructions)


load_keys(["OPENAI_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅


In [3]:
import os
from IPython.display import display, Image, Markdown
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import MessagesPlaceholder

# Initialize the model
chat_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

# Initialize the model (we use a slightly higher temperature for more creative/reasoning tasks)
exploring_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)


In [4]:
prompt_text = "Explain Quantum Computing like I'm five years old."
prompt = ChatPromptTemplate.from_template(prompt_text)
chain = prompt |  chat_model # | StrOutputParser()
reply = chain.invoke({})
reply

AIMessage(content='Okay! Imagine you have a magic coloring book. Normally, when you color a picture, you pick one color at a time—like red or blue. But this magic coloring book lets you color the picture with many colors all at once until you decide to look closely.\n\nQuantum computing is like that magic coloring book. Regular computers use tiny switches that are either ON or OFF, like red or blue. But quantum computers use special tiny things that can be ON, OFF, or both at the same time—like coloring with many colors at once! This helps them solve some puzzles much faster than regular computers.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 16, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 

In [5]:
prompt_text = "Explain Quantum Computing like I'm five years old."
prompt = ChatPromptTemplate.from_template(prompt_text)
chain = prompt |  chat_model | StrOutputParser()
reply = chain.invoke({})
reply

'Okay! Imagine you have a magic coloring book. Normally, when you color a picture, you have to choose one color at a time, like red or blue. But this magic coloring book lets you color the picture with many colors all at once!\n\nQuantum computing is like that magic coloring book. Regular computers think in bits, which are like tiny switches that can be either off (0) or on (1). But quantum computers use special tiny things called qubits that can be both off and on at the same time, like coloring with many colors at once. This helps quantum computers solve some problems much faster than regular computers!'

In [6]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-6-luna",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain Quantum Computing like I'm five years old."},
    ],
)
print(completion.choices[0].message.content)

A regular computer uses tiny switches that are either **off** or **on**—like **0** or **1**.

A quantum computer uses tiny things called **qubits**. Before you look at one, it can act a bit like a spinning coin: not simply heads or tails yet. A quantum computer carefully uses qubits so that, for some tricky problems, wrong possibilities fade away and useful ones become more likely.

When you check the answer, the qubits settle into ordinary 0s and 1s. Quantum computers aren’t better at every job—but they may be very helpful for certain special problems.


## 2. Chat models, message types and `ChatPromptTemplate`

A chat model does not take "a prompt" — it takes a **list of messages**, each with a role:

| Message | Who says it | Typical use |
|---|---|---|
| `SystemMessage` | you, the developer | role, rules, output format — the instructions that frame every turn |
| `HumanMessage` | the user | the question or the document to work on |
| `AIMessage` | the model | its earlier replies, when you send a conversation back |
| `ToolMessage` | a tool | the result of a tool call — you meet it in Module 5 |

`ChatPromptTemplate` builds that list for you from a template plus variables, and
`MessagesPlaceholder` reserves a slot where a whole conversation can be dropped in.

In [7]:
# The raw form: a list of typed messages.
conversation = [
    SystemMessage(content="You are a retail-banking assistant. Answer in one sentence."),
    HumanMessage(content="What does a credit score tell a lender?"),
]
reply = chat_model.invoke(conversation)
print(type(reply).__name__, "->", reply.content)
print("Token usage for this call:", reply.usage_metadata)

AIMessage -> A credit score tells a lender the likelihood that a borrower will repay their debts based on their credit history.
Token usage for this call: {'input_tokens': 33, 'output_tokens': 21, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [8]:
# The template form: roles fixed, content filled in per call, and a slot for history.
servicing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} at a retail bank. Answer in at most {sentence_limit} sentences."),
    MessagesPlaceholder("conversation_history"),
    ("human", "{customer_message}"),
])

prompt_values = {
    "role": "customer-service assistant",
    "sentence_limit": 2,
    "conversation_history": [
        HumanMessage(content="My card was declined at a fuel station this morning."),
        AIMessage(content="Sorry to hear that. Was it a debit card or a credit card?"),
    ],
    "customer_message": "Debit card. What should I check first?",
}

# See exactly what the model will receive...
for message in servicing_prompt.invoke(prompt_values).to_messages():
    print(f"{message.type:>6}: {message.content}")

system: You are a customer-service assistant at a retail bank. Answer in at most 2 sentences.
 human: My card was declined at a fuel station this morning.
    ai: Sorry to hear that. Was it a debit card or a credit card?
 human: Debit card. What should I check first?


In [9]:

# ...then run it as a chain: prompt | model | parser.
servicing_chain = servicing_prompt | chat_model | StrOutputParser()
print("\nReply:", servicing_chain.invoke(prompt_values))


Reply: Please check if your account has sufficient funds and if your card is active and not expired. If everything seems fine, contact our customer service to check for any holds or issues on your card.


In [10]:
conversation_history  = [
        HumanMessage(content="My card was declined at a fuel station this morning."),
        AIMessage(content="Sorry to hear that. Was it a debit card or a credit card?"),
    ]

In [11]:
response = servicing_chain.invoke(prompt_values)
response

'Please check if your account has sufficient funds and ensure your card is not blocked or expired. If everything seems fine, try contacting our customer service for further assistance.'

In [12]:
conversation_history = conversation_history + \
    [
        HumanMessage(content="Debit card. What should I check first?"),
        AIMessage(content=response),
    ]

In [13]:
conversation_history

[HumanMessage(content='My card was declined at a fuel station this morning.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sorry to hear that. Was it a debit card or a credit card?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Debit card. What should I check first?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Please check if your account has sufficient funds and ensure your card is not blocked or expired. If everything seems fine, try contacting our customer service for further assistance.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### LangChain, or just the provider's SDK?

Notice that **each SDK has a different shape** — this is the first thing every multi-provider team runs into:

| Provider | Client | "Chat" call |
|---|---|---|
| OpenAI | `OpenAI()` | `chat.completions.create(...)` |
| Anthropic | `Anthropic()` | `messages.create(...)` |
| Google | `genai.Client(...)` | `models.generate_content(...)` |

For **one call to one provider**, the raw SDK is enough — section 12 shows the OpenAI SDK returning a
validated Pydantic object in a handful of lines. LangChain earns its place when you need what a single
call does not give you: **composition** (`prompt | model | parser`), **swapping providers** without
rewriting code, **retries and fallbacks** (Lab 2.2), **tracing** in LangSmith, and plugging the
same pieces into **retrieval** (Modules 3–4) and **agents**
(Module 5 onward).

## 3. Tokens & context windows — the units a model actually sees

LLMs don't read characters or words — they read **tokens** (chunks of ~4 characters of English on average). Two consequences you must internalize:

1. **You pay per token** (input + output). Cost and latency scale with token count.
2. A model can only "see" a fixed number of tokens at once — its **context window**. Go past it and the earliest text is dropped.

Let's look at real tokens using OpenAI's `tiktoken`. (Each provider uses a *different* tokenizer, so token counts won't match exactly across providers — keep that in mind when comparing costs.)

In [14]:
import tiktoken

token_encoder = tiktoken.get_encoding("o200k_base")   # tokenizer family used by recent OpenAI models
sample_text = "ShopKart's Christmas sale starts Friday! Up to 60% off. 🎉"

token_ids = token_encoder.encode(sample_text)
print(f"Text        : {sample_text}")
print(f"Characters  : {len(sample_text)}")
print(f"Tokens      : {len(token_ids)}")
print(f"Token IDs   : {token_ids}")
print(f"Each token  : {[token_encoder.decode([token_id]) for token_id in token_ids]}")

Text        : ShopKart's Christmas sale starts Friday! Up to 60% off. 🎉
Characters  : 57
Tokens      : 17
Token IDs   : [19083, 179367, 885, 10431, 7357, 13217, 9377, 0, 4686, 316, 220, 1910, 4, 1277, 13, 139786, 231]
Each token  : ['Shop', 'Kart', "'s", ' Christmas', ' sale', ' starts', ' Friday', '!', ' Up', ' to', ' ', '60', '%', ' off', '.', ' �', '�']


**Read the output above:** notice how common words become single tokens, while emoji, rare words, and punctuation can split into several. Now compare the headline context windows (approximate — verify current values):

| Model | Context window (tokens) | Rough equivalent |
|---|---|---|
| `gpt-4.1-mini` | ~1,000,000 | a long book |
| `gemini-2.5-flash` | ~1,000,000 | a long book |
| `claude-haiku-4-5` | ~200,000 | a short book |

> **Business takeaway:** "Can I just paste the whole 300-page contract into the prompt?" Sometimes yes — but you pay for every token, every time. This is *why* Modules 3–4 (RAG) and the Module 7 take-home on context engineering exist.

## RTCCI FRAMEWORK FOR GOOD PROMPTS:

R: ROLE you want the AI model to assume

T: High level task for the AI Model (must be an action verb, i.e Summarize, draft, explain, analyse etc.)

C: Context: means the additional info required for the task

C: Constraints: output format, Do's and Don'ts, length of the oputput, tone of the output

I: Instructions: detialed instructions, step-by-step methodology you want the model to follow while performing that task.. (additional guidelines)

In [15]:

# Helper function to print results neatly
def execute_prompt(prompt_template, title, model=None, **kwargs):
    """
    Executes a given prompt template with context variables and prints the output.
    `model` defaults to chat_model; the Chain- and Tree-of-Thought cells pass exploring_model.
    """
    print(f"--- {title} ---")

    # Create the prompt using the template and input variables
    prompt = ChatPromptTemplate.from_template(prompt_template)

    # Create the chain
    chain = prompt | (model or chat_model) | StrOutputParser()

    # Invoke the chain with the provided context
    response = chain.invoke(kwargs)

    print("PROMPT:")
    print(prompt.format(**kwargs))
    print("\n" + "="*20 + " RESPONSE " + "="*20 + "\n")
    # print(response)
    display(Markdown(response))
    print("\n" + "-"*50 + "\n")

    return response

## **4. Crafting Precise Prompts with ChatPromptTemplate**

### **Scenario 1: Summarizing a Customer Loan Inquiry**

A loan officer needs a quick summary of a long customer email to assess eligibility at a glance.

#### **Bad Prompt Example 👎**

This prompt is too vague. It doesn't specify the output format or the key information to extract.

**Code:**

In [16]:
customer_email = """
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.
"""

bad_prompt_1 = "Summarize this email: {customer_email}"

_ = execute_prompt(bad_prompt_1, "Scenario 1: Bad Prompt", customer_email=customer_email)

--- Scenario 1: Bad Prompt ---
PROMPT:
Human: Summarize this email: 
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.


==================== RESPONSE ====================



Priya Sharma, a loyal customer of over 10 years and a software developer earning ₹18,00,000 annually, is inquiring about obtaining a ₹5,00,000 personal loan for a family wedding. She has a good credit history with a CIBIL score of 780, no existing loans with the bank, and a credit card with a ₹2,00,000 limit. She requests information on the next steps.


--------------------------------------------------



*Critique:* The summary is okay, but it's unstructured and mixes all the information. A loan officer would have to read through the paragraph to find specific data points.

#### **Good Prompt Example 👍**

This prompt is precise. It assigns a **role** (Loan Officer Assistant), provides clear **context**, specifies the **task**, and sets **constraints** on the output format (JSON).

**Code:**

In [17]:
customer_email = """
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.
"""

good_prompt_1 = """
You are a highly efficient AI assistant for a bank's loan officer.
Your task is to extract key information from a customer's loan inquiry email and present it in a structured JSON format.

**Email Content:**
---
{customer_email}
---

**Instructions:**
1. Extract the following fields: customer_name, loan_amount, loan_purpose, annual_salary, cibil_score, and existing_liabilities.
2. If a specific piece of information is not mentioned, use "N/A".
3. The output must be a valid JSON object. Do not add any introductory text or explanations.
"""

_ = execute_prompt(good_prompt_1, "Scenario 1: Good Prompt", customer_email=customer_email)

--- Scenario 1: Good Prompt ---
PROMPT:
Human: 
You are a highly efficient AI assistant for a bank's loan officer.
Your task is to extract key information from a customer's loan inquiry email and present it in a structured JSON format.

**Email Content:**
---

Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.

---

**Instructions:**
1. Extract the following fields: customer_name, loan_amount, loan_purpose, annual_salary, cibil_score, and existing_liabilities.
2. If a specific piece of information is not mentioned, use "N/A".
3. The output must be a valid J

```json
{
  "customer_name": "Priya Sharma",
  "loan_amount": "₹5,00,000",
  "loan_purpose": "family wedding",
  "annual_salary": "₹18,00,000",
  "cibil_score": 780,
  "existing_liabilities": "Credit card with ₹2,00,000 limit"
}
```


--------------------------------------------------



*Critique:* This response is perfect. The JSON format is machine-readable and easy for the loan officer to quickly parse. All key information is clearly labeled.

-----

### **Scenario 2: Drafting a Customer Communication for a Failed Transaction**

A customer's online payment failed. The bank needs to send a clear, empathetic, and helpful notification.

#### **Bad Prompt Example 👎**

This prompt lacks tone guidance and doesn't instruct the model on what helpful actions to suggest.

**Code:**

In [18]:
transaction_details = {
    "customer_name": "Mr. Verma",
    "amount": "₹3,500",
    "merchant": "Flipkart",
    "reason": "Incorrect CVV"
}

bad_prompt_2 = "Tell {customer_name} that their payment of {amount} to {merchant} failed because of {reason}."

_ = execute_prompt(bad_prompt_2, "Scenario 2: Bad Prompt", **transaction_details)

--- Scenario 2: Bad Prompt ---
PROMPT:
Human: Tell Mr. Verma that their payment of ₹3,500 to Flipkart failed because of Incorrect CVV.

==================== RESPONSE ====================



Sure! Here's a message you can send to Mr. Verma:

---

Dear Mr. Verma,

We would like to inform you that your payment of ₹3,500 to Flipkart was unsuccessful due to an incorrect CVV entered. Please verify your card details and try again.

If you need any assistance, feel free to reach out.

Thank you.

---

Would you like me to make it more formal or casual?


--------------------------------------------------




*Critique:* This response is blunt, robotic, and unhelpful. It could cause customer anxiety and doesn't offer a solution.

#### **Good Prompt Example 👍**

This prompt uses the **role-playing** technique, specifies the desired **tone** (empathetic and professional), and provides a clear **structure** with actionable next steps. This is a simple form of **few-shot prompting** by providing an example of the desired output style within the instructions.

**Code:**

In [19]:
transaction_details = {
    "customer_name": "Mr. Verma",
    "amount": "₹3,500",
    "merchant": "Flipkart",
    "reason": "Incorrect CVV"
}

good_prompt_2 = """
You are a Customer Service Bot for SecureBank.
Your task is to draft an SMS notification to a customer about a failed transaction.

**Tone:** Empathetic, professional, and reassuring.

**Instructions:**
1. Address the customer by name.
2. Clearly state that the transaction could not be processed.
3. Mention the amount and merchant.
4. Briefly and simply state the reason for the failure.
5. Provide a clear, immediate action the customer can take (e.g., "please try again, ensuring the details are correct").
6. End on a helpful note, providing a support channel.

**Customer and Transaction Details:**
- Customer Name: {customer_name}
- Amount: {amount}
- Merchant: {merchant}
- Reason: {reason}

Draft the SMS message.
"""

_ = execute_prompt(good_prompt_2, "Scenario 2: Good Prompt", **transaction_details)

--- Scenario 2: Good Prompt ---
PROMPT:
Human: 
You are a Customer Service Bot for SecureBank.
Your task is to draft an SMS notification to a customer about a failed transaction.

**Tone:** Empathetic, professional, and reassuring.

**Instructions:**
1. Address the customer by name.
2. Clearly state that the transaction could not be processed.
3. Mention the amount and merchant.
4. Briefly and simply state the reason for the failure.
5. Provide a clear, immediate action the customer can take (e.g., "please try again, ensuring the details are correct").
6. End on a helpful note, providing a support channel.

**Customer and Transaction Details:**
- Customer Name: Mr. Verma
- Amount: ₹3,500
- Merchant: Flipkart
- Reason: Incorrect CVV

Draft the SMS message.


==================== RESPONSE ====================



Dear Mr. Verma, your transaction of ₹3,500 at Flipkart could not be processed due to an incorrect CVV. Please try again, ensuring the CVV entered is correct. If you need assistance, contact SecureBank support at 1800-123-456. We're here to help.


--------------------------------------------------



*Critique:* This response is excellent. It's polite, explains the problem clearly, offers an immediate solution, and provides a way to get more help, building customer trust.

-----

### **Key Takeaways**

1.  **Be Specific:** Vague prompts lead to vague answers. Tell the model exactly what you want.
2.  **Assign a Role:** Telling the LLM to act as a "Loan Officer" or "Marketing Expert" primes it for better, context-aware responses.
3.  **Define the Format:** Don't leave the output structure to chance. Explicitly ask for JSON, a list, an email, or a table.
4.  **Set the Tone:** The same information can be delivered in many ways. Specify the tone (e.g., formal, empathetic, persuasive) to match the use case.
5.  **Provide Context:** The LLM doesn't know what you know. Give it all the necessary background information to complete the task successfully.

> ⏭️ **After the session.** This section is not run live in the workshop. Work through it afterwards, top to bottom.

## **5. The Persona Pattern**

**Background:** The **Persona Pattern** involves instructing the LLM to "act as" a specific role or character. This is one of the most effective ways to control the tone, style, and expertise level of the response. Assigning a persona primes the model to generate content that aligns with the assumed role's knowledge and communication style. 🎭

**Scenario:** A bank needs to draft a marketing email to a high-net-worth individual (HNI) about a new, exclusive investment fund. The tone must be professional, confident, and sophisticated.

In [20]:
fund_details = """
- Fund Name: "Apex Global Growth Fund"
- Strategy: Invests in high-growth international technology and healthcare equities.
- Target Audience: Sophisticated investors with a high risk appetite.
- Minimum Investment: ₹1 Crore.
"""

#### **Naive Prompt 👎**

Without a persona, the model's tone is generic and lacks the authority required for the target audience.

In [21]:
naive_persona_prompt = "Write an email about a new investment fund with these details:\n\n{details}"

_ = execute_prompt(
    naive_persona_prompt,
    "5.1 Naive Email Draft",
    details=fund_details
)

--- 5.1 Naive Email Draft ---
PROMPT:
Human: Write an email about a new investment fund with these details:


- Fund Name: "Apex Global Growth Fund"
- Strategy: Invests in high-growth international technology and healthcare equities.
- Target Audience: Sophisticated investors with a high risk appetite.
- Minimum Investment: ₹1 Crore.


==================== RESPONSE ====================



Subject: Introducing the Apex Global Growth Fund – Unlock High-Growth International Opportunities

Dear [Recipient's Name],

We are excited to announce the launch of our new investment vehicle, the **Apex Global Growth Fund**, designed exclusively for sophisticated investors seeking high-growth opportunities in the international markets.

**About the Fund:**  
The Apex Global Growth Fund focuses on investing in high-growth technology and healthcare equities across global markets. By targeting innovative companies at the forefront of their industries, the fund aims to deliver superior long-term capital appreciation.

**Who Should Invest:**  
This fund is tailored for investors with a high risk appetite who are looking to diversify their portfolio with exposure to dynamic sectors and geographies.

**Investment Details:**  
- **Minimum Investment:** ₹1 Crore  
- **Strategy:** Concentrated investments in international technology and healthcare equities with strong growth potential.

We believe the Apex Global Growth Fund offers a compelling opportunity to participate in the growth of transformative global industries.

For more information or to discuss how this fund fits into your investment strategy, please feel free to reach out.

Best regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]  
[Company Name]


--------------------------------------------------



*Critique:* This is dry, unprofessional, and fails to convey exclusivity or expertise. It would not appeal to a sophisticated investor.

#### **Persona Pattern Prompt 👍**

By assigning the persona of a **"Senior Wealth Manager,"** we instruct the model to adopt a suitable tone and frame the message appropriately.

In [22]:
persona_prompt = """
Act as a Senior Wealth Manager at a prestigious private bank.
Your task is to draft a personalized email to a long-standing, high-net-worth client.

The email should be sophisticated, concise, and compelling.
Introduce the new fund and suggest a brief meeting to discuss how it aligns with their portfolio goals.

Use the following fund details:
{details}
"""

_ = execute_prompt(
    persona_prompt,
    "5.2 Persona-Driven Email Draft",
    details=fund_details
)

--- 5.2 Persona-Driven Email Draft ---
PROMPT:
Human: 
Act as a Senior Wealth Manager at a prestigious private bank.
Your task is to draft a personalized email to a long-standing, high-net-worth client.

The email should be sophisticated, concise, and compelling.
Introduce the new fund and suggest a brief meeting to discuss how it aligns with their portfolio goals.

Use the following fund details:

- Fund Name: "Apex Global Growth Fund"
- Strategy: Invests in high-growth international technology and healthcare equities.
- Target Audience: Sophisticated investors with a high risk appetite.
- Minimum Investment: ₹1 Crore.



==================== RESPONSE ====================



Subject: Introducing the Apex Global Growth Fund – A Strategic Opportunity for Your Portfolio

Dear [Client’s Name],

I hope this message finds you well.

I am pleased to introduce the Apex Global Growth Fund, a new offering designed to capitalize on high-growth opportunities within international technology and healthcare equities. This fund is tailored for sophisticated investors with a high risk appetite, seeking to enhance portfolio diversification and capture long-term growth in dynamic sectors.

With a minimum investment of ₹1 Crore, the Apex Global Growth Fund presents a compelling avenue to align with your portfolio’s growth objectives.

I would welcome the opportunity to discuss how this fund can complement your existing investments. Please let me know a convenient time for a brief meeting.

Warm regards,  
[Your Full Name]  
Senior Wealth Manager  
[Private Bank Name]  
[Contact Information]


--------------------------------------------------



*Critique:* Excellent. The tone is professional, the language is sophisticated ("investment vehicle," "forward-thinking portfolio strategy"), and the call-to-action is respectful of the client's time.

## **6. The N-Shot Prompting Pattern**

**Background:** This pattern involves providing the model with examples ("shots") to guide its output.

  * **Zero-Shot:** You provide no examples, just the instruction. This works for simple, well-defined tasks.
  * **Few-Shot:** You provide a few examples of input and desired output. This is powerful for teaching the model a specific format, structure, or nuanced classification.

### **Zero-Shot Prompting: Instructing Without Examples**

#### **Background**

**Zero-shot prompting** is the most fundamental technique. It involves asking the model to perform a task for which it has received no specific, in-prompt examples. The model relies entirely on its pre-existing knowledge to understand the instruction and generate a response.

  * **How it works:** You give a direct command. For example, "Translate this text to French." The model has seen countless translation tasks during its training and knows what to do.
  * **Strengths:** Simple, fast, and effective for straightforward tasks that fall within the model's general capabilities (e.g., summarization, general classification, simple Q\&A).
  * **Weaknesses:** It can struggle with novel, highly specific, or nuanced tasks where the desired output format or logic is not obvious.
  * **Banking Use Cases:** Initial customer feedback analysis, categorizing support tickets into broad topics, drafting simple emails, identifying the general intent of a customer query.

#### **6.1 Zero-Shot Use Case: Customer Request Classification**

**Scenario:** A banking chatbot receives an incoming customer message. It needs to classify the message into a predefined category for routing to the correct department.

In [23]:
customer_message = "Hi, my credit card was charged twice for a purchase I made at BigBasket yesterday. Can you please help me fix this?"

**Improved Zero-Shot Prompt 👍**
We don't need examples here because the task is a straightforward classification. We just need to give clear instructions and the list of valid categories.

In [24]:
zeroshot_prompt = """
You are a classification bot.
Classify the following customer message into one of these categories:
- Account Inquiry
- Card Dispute
- Loan Application
- Technical Support

Return only the category name.

Message: "{message}"
"""

_ = execute_prompt(
    zeroshot_prompt,
    "6.1 Zero-Shot Classification",
    message=customer_message
)

--- 6.1 Zero-Shot Classification ---
PROMPT:
Human: 
You are a classification bot.
Classify the following customer message into one of these categories:
- Account Inquiry
- Card Dispute
- Loan Application
- Technical Support

Return only the category name.

Message: "Hi, my credit card was charged twice for a purchase I made at BigBasket yesterday. Can you please help me fix this?"


==================== RESPONSE ====================



Card Dispute


--------------------------------------------------



*Critique:* This is a perfect use of zero-shot prompting. The task is clear, and the model can easily perform it without needing examples.

### **Few-Shot Prompting: Learning from Examples**

#### **Background**

**Few-shot prompting** involves providing the LLM with a few examples ("shots") of the task you want it to perform directly within the prompt. This helps the model understand the desired pattern, format, and nuances.

  * **How it works:** You show the model input-output pairs. For instance, `Input: Sea otter -> Output: Enhydra lutris. Input: Manatee -> Output: Trichechus. Input: Blue whale -> Output:`. The model learns the pattern (common name to scientific name) and completes it.
  * **Strengths:** Extremely effective for tasks requiring a specific output structure (like JSON), a particular style, or classification into custom, non-obvious categories.
  * **Weaknesses:** Requires crafting high-quality examples, and makes the prompt longer and more expensive to process.
  * **Banking Use Cases:** Extracting specific data from unstructured documents (KYC, invoices), categorizing transactions into bank-specific buckets, formatting data for API calls, drafting communications that match a brand's voice.

#### **6.2 Few-Shot Use Case: Structuring Trade Data**

**Scenario:** A system needs to parse unstructured text from a trader's chat message and convert it into a structured format for the order management system. The format must be exact.

In [25]:
trader_chat = "we need to buy 5000 shares of RELIANCE at market price asap"

#### **Naive Prompt 👎**

A naive prompt will result in an unpredictable structure. The keys and values might change with every run.

In [26]:
naive_fewshot_prompt = "Extract the details from this trade message: {message}"

_ = execute_prompt(
    naive_fewshot_prompt,
    "6.2 Naive Trade Extraction",
    message=trader_chat
)

--- 6.2 Naive Trade Extraction ---
PROMPT:
Human: Extract the details from this trade message: we need to buy 5000 shares of RELIANCE at market price asap

==================== RESPONSE ====================



Here are the extracted details from the trade message:

- Action: Buy  
- Quantity: 5000 shares  
- Stock: RELIANCE  
- Price: Market price  
- Urgency: As soon as possible (ASAP)


--------------------------------------------------



*Critique:* This is just a string, not a machine-readable format. It's also missing a key for the price.

#### **Few-Shot Pattern Prompt 👍**

We provide a clear example to teach the model the **exact output format** we need.

In [27]:
fewshot_prompt = """
You are an AI that converts unstructured trader chat messages into a structured string.

Follow the format of the example precisely.
---
**Example:**
Message: "sell 200 INFY limit 1500"
Output: "ACTION:SELL|TICKER:INFY|QUANTITY:200|PRICE:1500.00"
---

**New Message:**
Message: "{message}"
Output:
"""

_ = execute_prompt(
    fewshot_prompt,
    "6.2 Few-Shot Trade Extraction",
    message=trader_chat
)

--- 6.2 Few-Shot Trade Extraction ---
PROMPT:
Human: 
You are an AI that converts unstructured trader chat messages into a structured string.

Follow the format of the example precisely.
---
**Example:**
Message: "sell 200 INFY limit 1500"
Output: "ACTION:SELL|TICKER:INFY|QUANTITY:200|PRICE:1500.00"
---

**New Message:**
Message: "we need to buy 5000 shares of RELIANCE at market price asap"
Output:


==================== RESPONSE ====================



Output: "ACTION:BUY|TICKER:RELIANCE|QUANTITY:5000|PRICE:MARKET"


--------------------------------------------------



*Critique:* Perfect. The few-shot example forced the model to produce an output that is structured, consistent, and ready for a downstream system to parse.

-----

---
### 6.3 Few-Shot at Scale with `FewShotChatMessagePromptTemplate`

The few-shot prompt above hard-codes its examples inside a single string. That's fine for 2-3 examples, but breaks down for production:
- You can't easily add/remove examples without re-editing the string
- You can't conditionally select examples per input (e.g., closest-match retrieval)
- You can't separately version-control your example library

LangChain's `FewShotChatMessagePromptTemplate` separates **examples** (data) from **prompt structure** (template). This is the production-scalable pattern.

In [29]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# 1. Examples live as data — easy to grow, swap, or load from JSON / DB.
examples = [
    {
        'input': 'we need to buy 5000 shares of RELIANCE at market price asap',
        'output': '''```json
{
  "action": "BUY",
  "ticker": "RELIANCE",
  "quantity": 5000,
  "order_type": "MARKET",
  "urgency": "HIGH"
}
```'''
    },
    {
        'input': 'sell 200 INFY at limit 1850',
        'output': '''```json
{
  "action": "SELL",
  "ticker": "INFY",
  "quantity": 200,
  "order_type": "LIMIT",
  "limit_price": 1850,
  "urgency": "NORMAL"
}
```'''
    },
]

# 2. Each example becomes a (Human, AI) message pair.
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai', '{output}'),
])

# 3. Wrap into a few-shot template — handles iteration over examples for you.
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 4. Assemble the final prompt — system + few-shot block + user input.
final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You convert short trader-chat strings to structured trade orders in JSON.'),
    few_shot_prompt,
    ('human', '{input}'),
])

print(final_prompt.format(input='buy 1000 TCS at market'))

System: You convert short trader-chat strings to structured trade orders in JSON.
Human: we need to buy 5000 shares of RELIANCE at market price asap
AI: ```json
{
  "action": "BUY",
  "ticker": "RELIANCE",
  "quantity": 5000,
  "order_type": "MARKET",
  "urgency": "HIGH"
}
```
Human: sell 200 INFY at limit 1850
AI: ```json
{
  "action": "SELL",
  "ticker": "INFY",
  "quantity": 200,
  "order_type": "LIMIT",
  "limit_price": 1850,
  "urgency": "NORMAL"
}
```
Human: buy 1000 TCS at market


In [30]:
# Run it on a new trader chat — model should follow the JSON schema established by the examples.
chain = final_prompt | chat_model | StrOutputParser()
result = chain.invoke({'input': 'sell 500 WIPRO at 520 limit, kind of urgent'})
print(result)

```json
{
  "action": "SELL",
  "ticker": "WIPRO",
  "quantity": 500,
  "order_type": "LIMIT",
  "limit_price": 520,
  "urgency": "HIGH"
}
```


#### 📝 Why this matters in production
- **Examples = data.** Store them in JSON / a database / Langfuse / LangSmith Hub and load at runtime.
- **Dynamic example selection.** Pair `FewShotChatMessagePromptTemplate` with a `SemanticSimilarityExampleSelector` to pick the *k* closest examples to the current input — significantly better than fixed examples for diverse inputs.
- **A/B testing.** Two example libraries → two chains → run both, compare quality.
- **Decoupling.** Prompt structure changes (system message, output format) don't require touching examples, and vice-versa.

---
## **7. The Directional Stimulus Pattern**

**Background:** This pattern guides the LLM to focus on specific aspects of a large piece of text. Instead of a generic summary, you provide "hints" or "directions" to tailor the output to your specific needs. 🎯

**Scenario:** A busy loan officer needs to quickly understand the key risks in a lengthy business credit report. They don't need a full summary, only the negative points.

In [35]:
credit_report_excerpt = """
Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY.
Their cash flow is positive, and they hold significant assets.
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers.
The market outlook for their sector is stable, but dependent on volatile international supply chains.
"""

#### **Naive Prompt 👎**

A generic summary will include positive points, forcing the officer to read through extra information.

In [36]:
naive_directional_prompt = "Summarize this credit report excerpt:\n{report}"

_ = execute_prompt(
    naive_directional_prompt,
    "7.1 Naive Report Summary",
    report=credit_report_excerpt
)

--- 7.1 Naive Report Summary ---
PROMPT:
Human: Summarize this credit report excerpt:

Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY.
Their cash flow is positive, and they hold significant assets.
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers.
The market outlook for their sector is stable, but dependent on volatile international supply chains.


==================== RESPONSE ====================



Future Gadgets Ltd. demonstrates strong financial performance with consistent 15% year-over-year revenue growth, positive cash flow, and substantial assets. However, the company has a high debt-to-equity ratio of 2.5 and has recently made two late payments to major suppliers. The sector's market outlook is stable but faces risks due to volatile international supply chains.


--------------------------------------------------



*Critique:* The summary is accurate but mixes positive and negative points.

#### **Directional Stimulus Prompt 👍**

We give the model a clear "hint" to focus *only* on the risk factors.

In [37]:
directional_prompt = """
Summarize the following credit report excerpt in 2-3 bullet points.

Hint: Focus exclusively on the negative points and potential risk factors mentioned in the report.

Report:
{report}
"""

_ = execute_prompt(
    directional_prompt,
    "7.2 Directional Risk Summary",
    report=credit_report_excerpt
)

--- 7.2 Directional Risk Summary ---
PROMPT:
Human: 
Summarize the following credit report excerpt in 2-3 bullet points.

Hint: Focus exclusively on the negative points and potential risk factors mentioned in the report.

Report:

Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY.
Their cash flow is positive, and they hold significant assets.
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers.
The market outlook for their sector is stable, but dependent on volatile international supply chains.



==================== RESPONSE ====================



- High debt-to-equity ratio of 2.5 indicates elevated financial leverage and potential risk.
- Two recent late payments to major suppliers suggest possible cash flow or management issues.
- Dependence on volatile international supply chains poses a risk to business stability despite a stable market outlook.


--------------------------------------------------



*Critique:* This is much more effective. It provides the loan officer with a concise, focused list of the exact information they need, saving them time and improving efficiency.

## **8. The Template & Meta Language Pattern**

**Background:** This powerful pattern involves defining a strict structure or a custom "language" for the model's output. By providing a template, you force the LLM to populate its response within your predefined format, making the output predictable and machine-readable. This is essential for JSON, XML, or other structured data needs. 📝

**Scenario:** A bank needs to generate a standardized risk assessment for a credit card application in a JSON format that can be logged directly into their system.

In [38]:
applicant_data = """
- Name: Rahul Verma
- Age: 28
- Annual Income: ₹8,00,000
- CIBIL Score: 760
- Existing Debt: ₹1,50,000 (Personal Loan)
"""

#### **Naive Prompt 👎**

Without a strict template, the model will generate JSON, but the structure, keys, and value types will be inconsistent.

In [39]:
naive_template_prompt = "Create a JSON risk summary for this credit card applicant:\n\n{data}"

_ = execute_prompt(
    naive_template_prompt,
    "8.1 Naive JSON Generation",
    data=applicant_data
)

--- 8.1 Naive JSON Generation ---
PROMPT:
Human: Create a JSON risk summary for this credit card applicant:


- Name: Rahul Verma
- Age: 28
- Annual Income: ₹8,00,000
- CIBIL Score: 760
- Existing Debt: ₹1,50,000 (Personal Loan)


==================== RESPONSE ====================



```json
{
  "applicant": {
    "name": "Rahul Verma",
    "age": 28,
    "annual_income_inr": 800000,
    "cibil_score": 760,
    "existing_debt_inr": 150000,
    "debt_to_income_ratio": 0.1875
  },
  "risk_summary": {
    "credit_score_risk": "Low risk - CIBIL score of 760 indicates good creditworthiness.",
    "debt_level_risk": "Moderate risk - Existing personal loan debt is 18.75% of annual income, which is manageable but should be monitored.",
    "age_factor": "Neutral - Age 28 is within typical range for credit card applicants.",
    "overall_risk_assessment": "Low to moderate risk. Applicant demonstrates good credit score and manageable debt levels."
  }
}
```


--------------------------------------------------



*Critique:* The keys (`applicant`, `income`, `risk`) are arbitrary, and the `risk` value is a subjective string, not a standardized category.

#### **Template Pattern Prompt 👍**

We define a "meta language" by specifying the exact keys and the allowed values for certain fields, ensuring a perfectly structured output.

In [40]:
template_prompt = """
Act as a risk assessment bot. Generate a JSON summary for the following credit card applicant.

The output must be a valid JSON object that strictly follows this template:
- "applicantId": string (use applicant's name)
- "cibilScore": integer
- "incomeToDebtRatio": float (calculate as Annual Income / Existing Debt)
- "riskLevel": string (must be one of the following: "Low", "Medium", "High")
- "isApproved": boolean (approve if score > 750 and incomeToDebtRatio > 5)

Applicant Data:
{data}
"""

_ = execute_prompt(
    template_prompt,
    "8.2 Template-Driven JSON Generation",
    data=applicant_data
)

--- 8.2 Template-Driven JSON Generation ---
PROMPT:
Human: 
Act as a risk assessment bot. Generate a JSON summary for the following credit card applicant.

The output must be a valid JSON object that strictly follows this template:
- "applicantId": string (use applicant's name)
- "cibilScore": integer
- "incomeToDebtRatio": float (calculate as Annual Income / Existing Debt)
- "riskLevel": string (must be one of the following: "Low", "Medium", "High")
- "isApproved": boolean (approve if score > 750 and incomeToDebtRatio > 5)

Applicant Data:

- Name: Rahul Verma
- Age: 28
- Annual Income: ₹8,00,000
- CIBIL Score: 760
- Existing Debt: ₹1,50,000 (Personal Loan)



==================== RESPONSE ====================



```json
{
  "applicantId": "Rahul Verma",
  "cibilScore": 760,
  "incomeToDebtRatio": 5.33,
  "riskLevel": "Low",
  "isApproved": true
}
```


--------------------------------------------------



*Critique:* This is perfect. The JSON is clean, follows the exact schema, performs a calculation as instructed, and uses a standardized value for `riskLevel`. This output can be reliably processed by another program.

## **9. Chain-of-Thought (CoT) Prompting: Showing Your Work**

### **Background**

**Chain-of-Thought (CoT) prompting** is a technique that encourages the model to break down a complex problem into intermediate steps before giving a final answer. By simply adding phrases like "Let's think step by step," you can significantly improve performance on tasks that require logic, reasoning, or calculation.

  * **How it works:** Instead of jumping to a conclusion, the model first "thinks" out loud, detailing its reasoning process. This mimics how humans solve complex problems and often leads the model to a more accurate result.
  * **Strengths:** Massively improves accuracy on arithmetic, commonsense, and symbolic reasoning tasks. The output is more transparent and auditable, as you can see *how* the model reached its conclusion.
  * **Banking Use Cases:** Assessing loan eligibility based on multiple criteria, preliminary fraud analysis, calculating financial ratios, explaining complex financial products to customers, and checking compliance with internal policies.


### **9.1. Chain-of-Thought (CoT): Evaluating a Small Business Loan**

**Scenario:** A loan officer needs a quick, reasoned assessment of a small business loan application to decide if it warrants a full underwriting process.

In [41]:
loan_application_data = """
- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720
"""

#### **Naive Prompt 👎**

A direct question will yield a "black box" answer with minimal justification, making it hard to trust or audit.

In [42]:
naive_prompt_3 = "Based on the data and criteria, should we approve a loan for Global Exports Inc.? \n{data}"

_ = execute_prompt(
    naive_prompt_3,
    "9.1 Naive Loan Assessment",
    model=exploring_model,
    data=loan_application_data
)

--- 9.1 Naive Loan Assessment ---
PROMPT:
Human: Based on the data and criteria, should we approve a loan for Global Exports Inc.? 

- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720


==================== RESPONSE ====================



Based on the provided data and the bank's minimum criteria:

- **Years in Business:** Global Exports Inc. has 2 years, but the minimum required is 3 years → **Does not meet criteria**
- **Annual Revenue:** ₹75,00,000, which is above the minimum ₹50,00,000 → **Meets criteria**
- **Owner's CIBIL Score:** 680, but the minimum required is 720 → **Does not meet criteria**

Since Global Exports Inc. does not meet the minimum years in business and the minimum CIBIL score requirements, **the loan should not be approved** based on the bank's criteria.


--------------------------------------------------




*Critique:* The answer is correct, but it lacks a structured breakdown of the decision process.

#### **Improved CoT Prompt 👍**

This prompt forces the model to perform a step-by-step analysis, comparing each application data point against the criteria before reaching a conclusion.

In [43]:
chain_of_thought_prompt_3 = """
You are a junior loan underwriter AI. Evaluate the small business loan application against the bank's criteria. 
Let's think step by step to create a clear evaluation report.

Application Data and Criteria:
{data}

**Evaluation Process:**
1.  **Check Years in Business:** Compare the applicant's years in business against the minimum requirement. State Pass or Fail.
2.  **Check Annual Revenue:** Compare the applicant's revenue against the minimum requirement. State Pass or Fail.
3.  **Check CIBIL Score:** Compare the owner's CIBIL score against the minimum requirement. State Pass or Fail.
4.  **Summary of Findings:** Briefly list the strengths and weaknesses of the application.
5.  **Final Recommendation:** Conclude with 'Proceed to Underwriting' or 'Reject'.
"""

_ = execute_prompt(
    chain_of_thought_prompt_3,
    "9.1 CoT Loan Assessment",
    model=exploring_model,
    data=loan_application_data
)

--- 9.1 CoT Loan Assessment ---
PROMPT:
Human: 
You are a junior loan underwriter AI. Evaluate the small business loan application against the bank's criteria. 
Let's think step by step to create a clear evaluation report.

Application Data and Criteria:

- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720


**Evaluation Process:**
1.  **Check Years in Business:** Compare the applicant's years in business against the minimum requirement. State Pass or Fail.
2.  **Check Annual Revenue:** Compare the applicant's revenue against the minimum requirement. State Pass or Fail.
3.  **Check CIBIL Score:** Compare the owner's CIBIL score against the minimum requirement. State Pass or Fail.
4.  **Summary of Findings:** Briefly list the strengths and weaknesses of the applica

**Evaluation Report for Loan Application: Global Exports Inc.**

---

**1. Check Years in Business:**  
- Applicant's Years in Business: 2  
- Minimum Required: 3  
- **Result:** Fail (Applicant has 1 year less than required)

---

**2. Check Annual Revenue:**  
- Applicant's Annual Revenue: ₹75,00,000  
- Minimum Required: ₹50,00,000  
- **Result:** Pass (Revenue exceeds minimum requirement)

---

**3. Check CIBIL Score:**  
- Applicant's CIBIL Score: 680  
- Minimum Required: 720  
- **Result:** Fail (Applicant's score is 40 points below minimum)

---

**4. Summary of Findings:**  
- **Strengths:**  
  - Annual revenue is strong and exceeds the minimum threshold.  
  - Requested loan amount (₹20,00,000) is reasonable relative to revenue.  
- **Weaknesses:**  
  - Business has only been operational for 2 years, below the minimum 3 years required.  
  - Owner's CIBIL score is below the minimum acceptable score, indicating higher credit risk.

---

**5. Final Recommendation:**  
- **Reject**  
Due to failure to meet the minimum years in business and CIBIL score criteria, the application does not satisfy the bank's minimum requirements for loan approval.


--------------------------------------------------




*Critique:* The response is transparent, auditable, and provides a clear, well-supported rationale for the decision.

### **Use Case 9.2: Preliminary Fraud Alert Analysis**

**Scenario:** A transaction is flagged for potential fraud. A junior analyst needs a summary explaining why it's suspicious.

#### **Naive Prompt 👎**

This prompt asks for a simple "yes/no" answer, which is not helpful and lacks any justification.

In [44]:
transaction_data_1 = """
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.
"""

naive_prompt_5 = "Is this transaction likely fraudulent? Transaction Details: {transaction}"

_ = execute_prompt(
    naive_prompt_5,
    "9.2 Naive Fraud Analysis",
    model=exploring_model,
    transaction=transaction_data_1
)

--- 9.2 Naive Fraud Analysis ---
PROMPT:
Human: Is this transaction likely fraudulent? Transaction Details: 
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.


==================== RESPONSE ====================



Based on the details provided, this transaction appears suspicious and could potentially be fraudulent. Here are the key points indicating risk:

- **Unusual Location:** The customer, Anjali Rao, usually transacts in Bangalore, India, but this transaction is in Prague, Czech Republic, which is a significant geographic deviation.
- **No Prior International Transactions:** There is no history of international spending, making this foreign transaction unexpected.
- **High Transaction Amount:** The amount (₹85,000) is substantially higher than the customer's average transaction value (₹2,500), indicating an unusual spending pattern.
- **Transaction Time:** The transaction occurred at 3:15 AM IST, which is an odd hour and may not align with the customer's typical activity.
- **Merchant Category:** Luxury Watches is a high-value category, often targeted in fraud.

**Recommendation:** Flag this transaction for further verification with the customer before approval to prevent potential fraud.


--------------------------------------------------



#### **Chain-of-Thought Prompt 👍**

We explicitly ask the model to reason through the evidence step-by-step, providing a clear, auditable analysis.

In [45]:
transaction_data_1 = """
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.
"""

chain_of_thought_prompt_1 = """
You are a fraud detection analyst.
Analyze the following transaction details to determine if it is likely fraudulent.
Let's think step by step to build your analysis before making a final conclusion.

Transaction Details:
{transaction}

First, identify all the risk factors. For each factor, explain why it is suspicious.
Second, summarize your findings.
Finally, conclude with a risk assessment: LOW, MEDIUM, or HIGH.
"""

_ = execute_prompt(
    chain_of_thought_prompt_1,
    "9.2 Chain-of-Thought Fraud Analysis",
    model=exploring_model,
    transaction=transaction_data_1
)

--- 9.2 Chain-of-Thought Fraud Analysis ---
PROMPT:
Human: 
You are a fraud detection analyst.
Analyze the following transaction details to determine if it is likely fraudulent.
Let's think step by step to build your analysis before making a final conclusion.

Transaction Details:

- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.


First, identify all the risk factors. For each factor, explain why it is suspicious.
Second, summarize your findings.
Finally, conclude with a risk assessment: LOW, MEDIUM, or HIGH.


==================== RESPONSE ====================



Let's analyze the transaction step by step.

---

### Step 1: Identify all the risk factors and explain why each is suspicious

1. **Transaction Location: Prague, CZ**
   - The customer, Anjali Rao, is usually located in Bangalore, India.
   - There is no prior history of international transactions.
   - A sudden international transaction, especially in a distant country, is unusual and raises suspicion of potential fraud or card compromise.

2. **Transaction Amount: ₹85,000**
   - The average transaction value for this customer is ₹2,500.
   - This transaction is more than 30 times the average transaction amount.
   - A sudden large transaction is a common red flag for fraud detection.

3. **Time of Transaction: 3:15 AM IST**
   - The transaction occurred at an odd hour (early morning).
   - Legitimate transactions, especially for luxury goods, are less likely to happen at this time.
   - Fraudsters often attempt transactions during off-hours to avoid immediate detection.

4. **Merchant Category: Luxury Watches**
   - Luxury goods are high-value items and often targeted in fraudulent transactions.
   - The customer’s usual spending pattern does not indicate purchases in luxury categories.

5. **Customer History: No prior international transactions**
   - Lack of prior international activity means this transaction is out of pattern.
   - Fraud detection systems often flag transactions that deviate significantly from historical behavior.

---

### Step 2: Summarize findings

- The transaction is international and in a location where the customer has no prior activity.
- The amount is significantly higher than the customer's usual spending.
- The transaction time is unusual.
- The merchant category is high-risk (luxury goods).
- The customer has no history of similar transactions.

All these factors combined strongly suggest that this transaction is out of character for the customer and could potentially be fraudulent.

---

### Step 3: Risk Assessment

**Risk Level: HIGH**

Given the multiple red flags—unusual location, high amount, odd timing, and high-risk merchant category—this transaction should be flagged for further investigation or immediate action such as contacting the customer or temporarily blocking the transaction.

---

**Final Recommendation:** Treat this transaction as high risk and initiate fraud prevention protocols.


--------------------------------------------------



## **10. Tree-of-Thought (ToT) Prompting: Exploring Multiple Paths**

### **Background**

**Tree-of-Thought (ToT)** is an advanced technique that generalizes Chain-of-Thought. Instead of just one linear reasoning path, ToT encourages the model to explore *multiple* different lines of reasoning (branches of a tree). It can then self-evaluate these paths and choose the most promising one to pursue.

  * **How it works:** A full ToT implementation is complex and often requires an agentic framework with multiple LLM calls. However, we can simulate the *spirit* of ToT in a single prompt by asking the model to:
    1.  **Generate** multiple potential solutions or viewpoints.
    2.  **Evaluate** the pros and cons of each.
    3.  **Select** the best option based on that evaluation.
  * **Strengths:** Exceptionally powerful for complex problems with no single right answer, such as strategic planning, creative problem-solving, and generating robust recommendations.
  * **Banking Use Cases:** Developing personalized financial plans for clients, creating customer retention strategies, brainstorming new product features, and formulating responses to complex regulatory inquiries.

### **10.1. Tree-of-Thought (ToT): Devising a Debt Restructuring Plan**

**Scenario:** A financial advisor is helping a client who is overwhelmed with multiple sources of debt. The goal is to propose and compare different concrete strategies for tackling the debt.

In [46]:
client_debt_profile = """
- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR
"""

#### **Naive Prompt 👎**

This prompt will generate generic, non-comparative advice that isn't tailored into actionable, alternative strategies.

In [47]:
naive_prompt_4 = "How can this client manage their debt? \n\n{profile}"

_ = execute_prompt(
    naive_prompt_4,
    "10.1 Naive Debt Advice",
    model=exploring_model,
    profile=client_debt_profile
)

--- 10.1 Naive Debt Advice ---
PROMPT:
Human: How can this client manage their debt? 


- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR


==================== RESPONSE ====================



To help Priya manage her debt effectively, the goal should be to minimize interest payments and pay off the highest-cost debt first while ensuring manageable monthly payments.

### Summary of Priya's Financials:
- **Monthly Take-home Salary:** ₹80,000  
- **Monthly Expenses (non-debt):** ₹45,000  
- **Available for Debt Repayment:** ₹35,000  
- **Debts:**
  1. Credit Card: ₹2,00,000 at 36% APR  
  2. Personal Loan: ₹5,00,000 at 14% APR  
  3. Car Loan: ₹3,00,000 at 9% APR  

---

## Step 1: Prioritize Debt Repayment by Interest Rate (Highest to Lowest)
1. **Credit Card (36%)** – Highest interest rate, very costly.
2. **Personal Loan (14%)** – Moderate interest.
3. **Car Loan (9%)** – Lowest interest rate.

---

## Step 2: Create a Debt Repayment Plan

### 1. Pay off Credit Card first
- Credit card debt is ₹2,00,000 at 36% APR, which is very expensive.
- Use as much of the ₹35,000 monthly repayment budget as possible to clear this quickly.
- Estimated monthly interest on ₹2,00,000 at 36% APR = ₹6,000 (approx).
- If Priya pays ₹35,000 monthly towards credit card:
  - ₹6,000 covers interest, ₹29,000 reduces principal.
  - Approximate payoff time: ~7-8 months.

### 2. After Credit Card is paid off, focus on Personal Loan
- Once credit card is cleared, redirect the entire ₹35,000 to personal loan repayment.
- Personal loan is ₹5,00,000 at 14% APR.
- Monthly interest on ₹5,00,000 at 14% APR = ₹5,833 approx.
- Paying ₹35,000 monthly means ₹29,167 goes to principal.
- Estimated payoff time: ~18-20 months.

### 3. Finally, pay off Car Loan
- After personal loan is cleared, use ₹35,000 to pay off car loan.
- Car loan ₹3,00,000 at 9% APR.
- Monthly interest approx ₹2,250.
- Paying ₹35,000 means ₹32,750 reduces principal.
- Estimated payoff time: ~9-10 months.

---

## Step 3: Additional Recommendations

- **Avoid new credit card debt:** Since credit card interest is very high, avoid accumulating more balance.
- **Consider balance transfer or refinancing:** If possible, transfer credit card balance to a personal loan with lower interest or negotiate lower rates.
- **Build an emergency fund:** Once debts are cleared, save to avoid future debt.
- **Track expenses:** Ensure ₹45,000 monthly expenses are optimized to maintain ₹35,000 for debt repayment.

---

## Summary Table

| Debt           | Amount (₹) | Interest Rate | Priority | Monthly Payment (₹) | Estimated Payoff Time |
|----------------|------------|---------------|----------|---------------------|-----------------------|
| Credit Card    | 2,00,000   | 36%           | 1        | 35,000              | ~7-8 months           |
| Personal Loan  | 5,00,000   | 14%           | 2        | 35,000              | ~18-20 months         |
| Car Loan       | 3,00,000   | 9%            | 3        | 35,000              | ~9-10 months          |

---

### Conclusion:
Priya should aggressively pay off the credit card debt first using the ₹35,000 monthly repayment amount, then move to the personal loan, and finally the car loan. This approach minimizes interest payments and clears debt efficiently.


--------------------------------------------------



*Critique:* The advice is sound but lacks structure and fails to explore or compare alternative, well-known strategies.

#### **Improved "ToT-style" Prompt 👍**

This prompt instructs the model to generate and evaluate multiple distinct strategies, presenting a comprehensive analysis of the options.

In [48]:
print(client_debt_profile)


- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR



In [49]:
tree_of_thought_prompt_4 = """
You are an expert financial advisor. 
For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** 
Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** 
For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** 
Based on your evaluation, select the single most suitable strategy and explain why it is the best recommendation for someone in this situation.
"""

_ = execute_prompt(
    tree_of_thought_prompt_4,
    "10.1 ToT-style Debt Plan",
    model=exploring_model,
    profile=client_debt_profile
)

--- 10.1 ToT-style Debt Plan ---
PROMPT:
Human: 
You are an expert financial advisor. 
For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:

- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR


1.  **Propose Three Distinct Strategies:** 
Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** 
For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** 
Based on your evaluation, select the single most suitable stra

Certainly! Here's a structured debt management plan for Priya based on her profile and debts.

---

### 1. Propose Three Distinct Strategies

#### Strategy A: Debt Avalanche
- **How it works:** Priya allocates her available ₹35,000 monthly debt repayment to the debt with the highest interest rate first (Credit Card at 36%), while making minimum payments on the other two loans. Once the credit card is fully paid off, she moves to the Personal Loan (14%), then finally the Car Loan (9%).
  
#### Strategy B: Debt Snowball
- **How it works:** Priya focuses on paying off the smallest debt first to gain psychological momentum. She pays off the Credit Card (₹2,00,000) first since it’s the smallest, then the Car Loan (₹3,00,000), and finally the Personal Loan (₹5,00,000), regardless of interest rates. Minimum payments are made on other debts while focusing on one.

#### Strategy C: Loan Consolidation
- **How it works:** Priya takes a new personal loan or balance transfer at a lower interest rate (ideally below 14%) to pay off the high-interest credit card debt and possibly the car loan. She then repays this consolidated loan at a single, lower interest rate with a fixed monthly payment of ₹35,000.

---

### 2. Evaluate Each Strategy

| Strategy           | Primary Pro                                             | Primary Con                                              |
|--------------------|---------------------------------------------------------|----------------------------------------------------------|
| **Debt Avalanche** | Minimizes total interest paid over time, saving money. | Takes longer to see smaller debts cleared, which may feel discouraging. |
| **Debt Snowball**  | Provides quick wins by paying off smaller debts first, boosting motivation. | Results in higher total interest paid due to ignoring interest rates. |
| **Loan Consolidation** | Simplifies payments and reduces interest rate on high-cost debt. | Requires good credit score and may involve fees; interest rate may not be low enough to justify. |

---

### 3. Recommend and Justify

**Recommended Strategy: Debt Avalanche**

**Justification:**

- Priya’s highest interest debt is the Credit Card at 36%, which is extremely costly. Paying this off first will drastically reduce the interest burden.
- She has a healthy monthly surplus of ₹35,000 for debt repayment, so focusing on the highest interest debt will save her the most money in the long run.
- Although the Debt Snowball offers psychological benefits, the high interest on the credit card means delaying its repayment will cost her significantly more.
- Loan consolidation could be beneficial only if Priya can secure a significantly lower interest rate (below 14%) and minimal fees. Otherwise, the cost and effort may not justify it.
- Therefore, the Debt Avalanche balances financial prudence with her capacity to repay, ensuring she pays the least interest overall and becomes debt-free faster.

---

If Priya needs help with budgeting or negotiating with lenders for better terms, those can be additional steps after choosing the Debt Avalanche strategy.


--------------------------------------------------




*Critique:* This response provides a strategic, multi-faceted analysis that empowers the client and advisor to make an informed decision based on clear trade-offs.

### **Use Case 10.2: Proposing a Client Investment Strategy**

**Scenario:** A financial advisor needs to propose an investment strategy for a young client with a moderate risk appetite.

#### **Naive Prompt 👎**

This gives a generic, one-size-fits-all answer without considering alternatives or trade-offs.

In [50]:
client_profile_1 = """
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000
"""

naive_prompt_6 = "Suggest an investment portfolio for this client: {profile}"

_ = execute_prompt(
    naive_prompt_6,
    "10.2 Naive Investment Suggestion",
    model=exploring_model,
    profile=client_profile_1
)

--- 10.2 Naive Investment Suggestion ---
PROMPT:
Human: Suggest an investment portfolio for this client: 
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000


==================== RESPONSE ====================



Certainly! For Sameer Khan, age 28, with a moderate risk tolerance and a goal of wealth accumulation for early retirement, here’s a suggested diversified investment portfolio. The focus will be on growth-oriented assets with a balanced approach to risk.

---

### Investment Portfolio Suggestion for Sameer Khan

**Client Profile:**
- Age: 28
- Goal: Wealth accumulation for early retirement (say around 45-50 years)
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000

---

### Key Considerations:
- **Time Horizon:** ~20+ years (ample time to benefit from compounding)
- **Moderate Risk:** Balanced between equity growth and debt stability
- **Tax Efficiency:** Use tax-advantaged instruments where possible
- **Liquidity:** Maintain some liquid assets for emergencies

---

### Suggested Asset Allocation

| Asset Class               | Allocation (%) | Rationale                                                                                      |
|--------------------------|----------------|------------------------------------------------------------------------------------------------|
| Equity (Stocks & Equity Mutual Funds) | 55%            | Growth engine of the portfolio; includes large-cap, mid-cap, and index funds for diversification |
| Debt Instruments (Bonds, Debt Funds, PPF) | 30%            | Stability and capital preservation; helps reduce portfolio volatility                            |
| Alternative Investments (Gold, REITs) | 10%            | Hedge against inflation and diversification                                                     |
| Cash & Liquid Funds       | 5%             | Emergency fund and liquidity                                                                   |

---

### Detailed Investment Suggestions

#### 1. **Equity (55%)**
- **Large-cap Mutual Funds / ETFs:** 30%
  - Examples: SBI Bluechip Fund, HDFC Top 100, Nifty 50 ETF
- **Mid-cap / Multi-cap Funds:** 15%
  - Examples: Axis Midcap Fund, Kotak Emerging Equity Fund
- **Direct Stocks (Optional):** 10%
  - Focus on quality companies with good growth potential and dividends

#### 2. **Debt Instruments (30%)**
- **Public Provident Fund (PPF):** 10%
  - Tax-free returns, 15-year lock-in, good for long-term debt allocation
- **Debt Mutual Funds (Short-term or Corporate Bond Funds):** 15%
  - Examples: HDFC Corporate Bond Fund, ICICI Prudential Short Term Fund
- **Fixed Deposits / Recurring Deposits:** 5%
  - For safety and predictable returns

#### 3. **Alternative Investments (10%)**
- **Gold (via Sovereign Gold Bonds or Gold ETFs):** 7%
  - Inflation hedge and portfolio diversification
- **REITs (Real Estate Investment Trusts):** 3%
  - Exposure to real estate without direct property investment

#### 4. **Cash & Liquid Funds (5%)**
- Maintain an emergency fund equivalent to 6 months of expenses in a liquid fund or savings account.

---

### Additional Recommendations

- **Systematic Investment Plan (SIP):** Invest monthly via SIPs in mutual funds to average out market volatility.
- **Tax Planning:** Maximize deductions under Section 80C (PPF, ELSS funds) and consider NPS for additional tax benefits.
- **Review & Rebalance:** Annually review portfolio to rebalance back to target allocation.
- **Insurance:** Ensure adequate health and term insurance coverage to protect wealth accumulation.

---

### Sample Monthly Investment Plan (Assuming 30% savings rate = ₹7,50,000/year ≈ ₹62,500/month)

| Asset Class               | Monthly SIP Amount (₹) | Fund/Instrument Examples                     |
|--------------------------|-----------------------|----------------------------------------------|
| Equity                   | ₹34,375               | Large-cap + Mid-cap mutual funds              |
| Debt                     | ₹18,750               | PPF + Debt mutual funds                        |
| Gold                     | ₹4,375                | Sovereign Gold Bonds or Gold ETFs             |
| Cash / Liquid Funds      | ₹3,125                | Liquid mutual funds for emergencies           |

---

If you want, I can also help suggest specific funds or stocks based on current market conditions. Would you like that?


--------------------------------------------------



#### **"ToT-style" Prompt 👍**

This prompt guides the model to explore and critique several strategies before making a final, well-justified recommendation.

In [51]:
client_profile_1 = """
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000
"""

tree_of_thought_prompt_1 = """
You are a senior investment strategist. Your task is to devise an investment strategy for a new client.
Use the following structured thinking process:

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** Based on the client's profile, generate three different investment strategies. Name each one (e.g., 'Balanced Growth', 'Aggressive Tech Focus', 'Diversified Core'). For each, suggest a sample asset allocation (Equities, Debt, Alternatives).
2.  **Evaluate Each Strategy:** For each of the three strategies, briefly list the potential Pros and Cons specifically for this client. Consider their age, goal, and risk tolerance.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy for the client and write a concluding paragraph explaining why it is the best choice.
"""

_ = execute_prompt(
    tree_of_thought_prompt_1,
    "10.2 ToT-style Investment Strategy",
    model=exploring_model,
    profile=client_profile_1
)

--- 10.2 ToT-style Investment Strategy ---
PROMPT:
Human: 
You are a senior investment strategist. Your task is to devise an investment strategy for a new client.
Use the following structured thinking process:

Client Profile:

- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000


1.  **Propose Three Distinct Strategies:** Based on the client's profile, generate three different investment strategies. Name each one (e.g., 'Balanced Growth', 'Aggressive Tech Focus', 'Diversified Core'). For each, suggest a sample asset allocation (Equities, Debt, Alternatives).
2.  **Evaluate Each Strategy:** For each of the three strategies, briefly list the potential Pros and Cons specifically for this client. Consider their age, goal, and risk tolerance.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy for the client and write a concluding paragraph explaining why it is th

Certainly! Below is a structured investment strategy proposal for Sameer Khan based on his profile.

---

### Client Profile Recap:
- Name: Sameer Khan  
- Age: 28  
- Goal: Wealth accumulation for early retirement (likely 10-15 years horizon)  
- Risk Tolerance: Moderate  
- Annual Income: ₹25,00,000  

---

## 1. Propose Three Distinct Strategies

### Strategy 1: Balanced Growth  
- **Equities:** 60% (Mix of large-cap, mid-cap, and some international equities)  
- **Debt:** 30% (Government bonds, corporate bonds, fixed deposits)  
- **Alternatives:** 10% (Real estate investment trusts (REITs), gold, and some exposure to private equity or startups via funds)  

### Strategy 2: Aggressive Tech Focus  
- **Equities:** 80% (Heavy focus on technology, innovation-driven sectors, including Indian and global tech stocks and ETFs)  
- **Debt:** 10% (Short-term debt instruments for liquidity and risk mitigation)  
- **Alternatives:** 10% (Cryptocurrency, venture capital funds, thematic funds in emerging technologies)  

### Strategy 3: Diversified Core  
- **Equities:** 50% (Broad market index funds, blue-chip stocks, and dividend-paying stocks)  
- **Debt:** 40% (Mix of government securities, corporate bonds, and fixed income mutual funds)  
- **Alternatives:** 10% (Gold, real estate, and balanced hybrid funds)  

---

## 2. Evaluate Each Strategy

### Strategy 1: Balanced Growth  
**Pros:**  
- Good equity exposure to capture growth potential at a young age.  
- Debt allocation reduces volatility and provides stability.  
- Alternatives add diversification and inflation hedge.  
- Aligns well with moderate risk tolerance.  

**Cons:**  
- May deliver moderate returns, possibly slower wealth accumulation compared to aggressive strategies.  
- Requires active monitoring and rebalancing.  

---

### Strategy 2: Aggressive Tech Focus  
**Pros:**  
- High growth potential due to focus on technology and innovation sectors.  
- Suitable for a young investor with a long horizon.  
- Could accelerate wealth accumulation for early retirement.  

**Cons:**  
- High volatility and risk, may cause emotional stress given moderate risk tolerance.  
- Concentrated sector risk; tech sector downturns could significantly impact portfolio.  
- Less stable income streams, higher chance of drawdowns.  

---

### Strategy 3: Diversified Core  
**Pros:**  
- Very stable and diversified across asset classes.  
- Lower volatility, suitable for moderate risk tolerance.  
- Regular income from dividend stocks and bonds.  
- Easier to manage and less stressful for the client.  

**Cons:**  
- Lower equity allocation may limit growth potential.  
- May not accumulate sufficient wealth quickly enough for early retirement goals.  
- Alternatives allocation is conservative, limiting inflation hedging.  

---

## 3. Recommend and Justify

**Recommended Strategy: Balanced Growth**

Given Sameer’s age (28), goal of early retirement, and moderate risk tolerance, the **Balanced Growth** strategy offers the best compromise between growth and risk management. It provides sufficient equity exposure to harness the power of compounding over a 10-15 year horizon while maintaining a meaningful allocation to debt to cushion against market volatility. The inclusion of alternatives like gold and REITs further diversifies the portfolio and provides protection against inflation and market downturns. This approach aligns well with Sameer’s moderate risk appetite and wealth accumulation goal, offering a structured yet flexible framework that can be adjusted as his circumstances evolve.

---

If you would like, I can also suggest specific mutual funds, ETFs, or direct stock picks aligned with this strategy.


--------------------------------------------------



## 11. Choosing a pattern — which pattern for which task

### Foundational Prompt Patterns

| Concept | Takeaway |
|---|---|
| **Persona pattern** | Sets tone, expertise, audience — must be in the system prompt |
| **N-shot pattern** | Lock format and decision boundaries the model can't infer from instructions |
| **Directional stimulus** | Direct attention to specific aspects of input — e.g. 'focus on the negative factors' |
| **Template pattern** | Enforce schema explicitly so downstream code can parse without surprises |
| **`FewShotChatMessagePromptTemplate`** | The production scalable form — examples as data, prompt as template |

### Advanced Prompting — Zero-Shot, Few-Shot, CoT, ToT, CoVe

| Concept | Takeaway |
|---|---|
| **Zero-shot vs few-shot** | Zero-shot for simple tasks on capable models; few-shot when format is unusual or accuracy matters |
| **Chain-of-Thought (CoT)** | Verbalize reasoning before answer — auditable, accurate, slower |
| **Tree-of-Thought (ToT)** | Explore multiple hypotheses in parallel and pick the best — for ambiguous, multi-strategy decisions |
| **Chain-of-Verification (CoVe)** | Draft → plan questions → answer independently → revise — substantial hallucination reduction |
| **Reasoning models change the math** | On `gpt-5-mini` / `o1` you don't need 'let's think step by step' — internal CoT happens automatically |
| **Cost / latency trade-off** | CoT is ~2× longer output; CoVe is ~3-4× total tokens. Worth it when accuracy matters more than speed |

## 12. Structured outputs with Pydantic — the headline skill ⭐

So far every answer was **free text**. Production systems need **structured, machine-readable** data they can route, store, and act on. Asking a model to "return JSON" is fragile (it adds prose, breaks quotes, drifts schema). Instead we define a **Pydantic schema** and let the provider **guarantee** the output matches it.

**Business scenario — cross-industry support triage.** An inbound message could be from banking, healthcare, e-commerce, insurance... The intake system must classify each into a fixed schema so it can auto-route to the right queue and dashboard.

In [52]:
from pydantic import BaseModel, Field
from typing import Literal, List

class SupportTicket(BaseModel):
    """Structured triage for any inbound customer message."""
    category: Literal["billing", "technical", "complaint", "inquiry", "fraud_or_safety", "cancellation"] = Field(
        description="Primary intent of the message")
    urgency: Literal["low", "medium", "high", "critical"] = Field(description="How fast a human must act")
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    industry: str = Field(description="Best-guess industry, e.g. 'banking', 'healthcare', 'e-commerce'")
    entities: List[str] = Field(description="Key entities: order ids, amounts, names, dates")
    summary: str = Field(description="One-sentence summary")
    recommended_action: str = Field(description="Suggested next step for the support agent")

### 12a · OpenAI `responses.parse()` — guaranteed schema → a Pydantic object

In [53]:
from openai import OpenAI

openai_client = OpenAI()

message_text = ("My credit card was charged ₹18,250 TWICE for the same order #A4821 this morning. "
                "I need one charge refunded today — this is unacceptable.")

response = openai_client.responses.parse(
    model="gpt-5-mini",
    input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
           {"role": "user", "content": message_text}],
    text_format=SupportTicket,
)
ticket = response.output_parsed          # <-- a real, validated SupportTicket instance
print(type(ticket).__name__, "->")
print(ticket.model_dump_json(indent=2))

SupportTicket ->
{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "₹18,250",
    "Order #A4821",
    "charged twice",
    "this morning",
    "credit card"
  ],
  "summary": "Customer reports being charged ₹18,250 twice for order #A4821 and requests one charge refunded today.",
  "recommended_action": "Immediately check payment/transaction logs for order #A4821 to confirm duplicate charge; if confirmed, initiate refund for one of the charges and provide the customer a refund confirmation and expected timeline. If transaction details are missing, request the card's last 4 digits or transaction IDs and escalate to billing/payments team for urgent processing."
}


In [54]:
print(response.output_parsed)

category='billing' urgency='high' sentiment='angry' industry='e-commerce' entities=['₹18,250', 'Order #A4821', 'charged twice', 'this morning', 'credit card'] summary='Customer reports being charged ₹18,250 twice for order #A4821 and requests one charge refunded today.' recommended_action="Immediately check payment/transaction logs for order #A4821 to confirm duplicate charge; if confirmed, initiate refund for one of the charges and provide the customer a refund confirmation and expected timeline. If transaction details are missing, request the card's last 4 digits or transaction IDs and escalate to billing/payments team for urgent processing."


### 12b · The LangChain way — `with_structured_output`

The same schema, through LangChain: `with_structured_output(SupportTicket)` makes the chain return a
validated `SupportTicket` object instead of text, so it composes with everything else in this workshop.
`gpt-5-mini` is a reasoning model, so it is created without a temperature:

**Key rule:** reasoning models typically **don't take a temperature** — you steer them with *effort*, not randomness.

In [55]:
# The LangChain way: the model returns a validated SupportTicket object, not text.
reasoning_model = ChatOpenAI(model="gpt-5-mini")        # reasoning model: no temperature
triage_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the customer message into the SupportTicket schema."),
    ("human", "{message}"),
])
triage_chain = triage_prompt | reasoning_model.with_structured_output(SupportTicket)

ticket = triage_chain.invoke({"message": message_text})
print(type(ticket).__name__, "->")
print(ticket.model_dump_json(indent=2))
print("\nRoute on a field, no string parsing:", "ESCALATE" if ticket.urgency in ("high", "critical") else "queue")

SupportTicket ->
{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "₹18,250",
    "order #A4821",
    "charged twice",
    "this morning",
    "credit card"
  ],
  "summary": "Customer reports being double-charged ₹18,250 for order #A4821 this morning and demands one of the charges be refunded today.",
  "recommended_action": "Immediately verify payments for order #A4821 and confirm the duplicate charge. If confirmed, process a same-day refund for the duplicate charge and send the customer refund confirmation (transaction ID and expected posting time). If same-day refund is not possible, escalate to billing supervisor, notify the customer of the timeline, and request any needed details (last four digits of card or transaction IDs)."
}

Route on a field, no string parsing: ESCALATE


### 12c · Classify a batch of diverse, multi-industry messages

In [56]:
import pandas as pd

messages = [
    "The MRI report upload on your patient portal keeps failing with error 500. I need it before my appointment tomorrow.",   # healthcare
    "I got an OTP I never requested and someone tried to log into my account. Please block it NOW.",                          # fraud / banking
    "Just wanted to say your delivery partner was super polite and my package arrived a day early. Thank you!",               # e-commerce / positive
    "I want to cancel my annual insurance policy renewal scheduled next week and get a pro-rata refund.",                     # insurance
    "What are your customer-care hours on public holidays?",                                                                  # generic inquiry
]

rows = []
for inbound_message in messages:
    parse_response = openai_client.responses.parse(
        model="gpt-5-mini",
        input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
               {"role": "user", "content": inbound_message}],
        text_format=SupportTicket,
    )
    triaged_ticket = parse_response.output_parsed
    rows.append({"industry": triaged_ticket.industry, "category": triaged_ticket.category,
                 "urgency": triaged_ticket.urgency,
                 "sentiment": triaged_ticket.sentiment,
                 "action": triaged_ticket.recommended_action[:45]})

print(pd.DataFrame(rows).to_string(index=False))

       industry        category  urgency sentiment                                        action
     healthcare       technical     high  negative Immediately escalate to technical support/eng
online services fraud_or_safety critical     angry Immediately suspend/block the account and inv
     e-commerce         inquiry      low  positive Acknowledge the customer's feedback with a th
      insurance    cancellation     high   neutral Request policy number and account details, ca
        general         inquiry      low   neutral Provide the customer-care hours for public ho


> Every row is a **validated object** — `ticket.urgency` is always one of the allowed values. You can now route on `urgency == "critical"` with zero string-parsing. *This* is why structured output is the backbone of agents (Module 5 onward).

### 12d · Structured output across providers — who supports what

Native structured-output support varies. Know the fallbacks:

| Provider | Native schema enforcement | Method |
|---|:--:|---|
| OpenAI | ✅ | `responses.parse(text_format=Model)` |
| Google Gemini | ✅ | `GenerateContentConfig(response_schema=Model)` → `resp.parsed` |
| Anthropic Claude | ⚠️ (via tool/prompt) | prompt for JSON, then **validate with Pydantic** |
| Groq (Llama) | ⚠️ | `response_format={"type": "json_object"}` + Pydantic validate |

The universal safety net is the same everywhere: **parse the text, then `Model.model_validate_json(...)`** so bad output fails loudly instead of silently corrupting downstream systems.

### When the output does not match — validation failure, then retry

A schema is only useful if a **bad** payload is stopped before it reaches the case system. Pydantic does the
stopping; a bounded retry gives the model another chance.

In [57]:
from langchain_core.exceptions import OutputParserException
from pydantic import ValidationError

# A payload that breaks the schema: "urgency" must be one of low / medium / high / critical.
malformed_payload = ('{"category": "billing", "urgency": "extreme", "sentiment": "angry", '
                     '"industry": "banking", "entities": ["18,250"], '
                     '"summary": "Duplicate card charge.", "recommended_action": "Refund one charge."}')
try:
    SupportTicket.model_validate_json(malformed_payload)
except ValidationError as validation_error:
    print("Rejected before it reached any downstream system:")
    for problem in validation_error.errors():
        print(f"  field {problem['loc']}: {problem['msg']}")

# OpenAI enforces the schema on its side, so a live mismatch is rare here. Providers that only
# *prompt* for JSON mismatch far more often. The defence is the same everywhere: validate, and
# retry a bounded number of times. `.with_retry()` works on any Runnable — Lab 2.2 returns to it.
resilient_triage_chain = triage_chain.with_retry(
    retry_if_exception_type=(ValidationError, OutputParserException), stop_after_attempt=3)
retried_ticket = resilient_triage_chain.invoke(
    {"message": "I got an OTP I never requested and someone tried to log into my account. Please block it NOW."})
print("\nWith a bounded retry in place ->", retried_ticket.category, "/", retried_ticket.urgency)

Rejected before it reached any downstream system:
  field ('urgency',): Input should be 'low', 'medium', 'high' or 'critical'

With a bounded retry in place -> fraud_or_safety / critical


## 14. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Role + context + constraints** | The 3 dimensions every prompt should explicitly set |
| **ChatPromptTemplate** | Replaces raw f-strings — composable, parameterizable, testable |
| Messages, `ChatPromptTemplate`, `MessagesPlaceholder` | Every chain, retriever and agent in this workshop starts here |
| Inspected tokens & context windows | You pay per token; context is finite |
| **Few-shot examples** | Force tone, format, and decision boundaries the model can't infer from instructions alone |
| **Critique-then-rewrite** | Read every output critically; the second prompt is always better than the first |
| **`prompt | llm` chains** | Foundation for every chain in Modules 2–9 |
| **Pydantic structured outputs** | Validated objects, not fragile JSON-in-text — the backbone of agents |
| Cross-provider structured output | Know each provider's native support and the universal validate-it fallback |
| Validation failure and a bounded retry | Bad output is stopped, not stored |

**Next →** Module 2, Lab 2.1 — **LangChain Fundamentals**: composing these prompts into LCEL chains, parsing, streaming and running branches in parallel.